### This notebook inspects the two Econ marts:
#### - data/marts/fct_macroecon_flows_yearly.csv
#### - data/marts/fct_trade_by_naics_yearly.parquet

In [ ]:
from pathlib import Path
import os
import pandas as pd

try:
    import duckdb  # fast & memory-efficient for big CSVs
    HAVE_DUCKDB = True
except Exception:
    HAVE_DUCKDB = False

pd.set_option("display.max_columns", None)   # show all columns
pd.set_option("display.max_colwidth", None)  # don't truncate cell values
pd.set_option("display.width", None) 
pd.set_option("display.max_seq_items", None)  # don't abbreviate lists/tuples in cells

In [ ]:
# Locate the files - navigate up from notebooks folder to project root
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_MARTS = BASE_DIR / "data" / "marts"

FACT_ECON = DATA_MARTS / "fct_macroecon_flows_yearly.csv"
FACT_TRADE = DATA_MARTS / "fct_trade_by_naics_yearly.parquet"

print("ECON exists:", FACT_ECON.exists(), "->", FACT_ECON)
print("TRADE exists:", FACT_TRADE.exists(), "->", FACT_TRADE)

assert FACT_ECON.exists(), f"Missing: {FACT_ECON}"
assert FACT_TRADE.exists(), f"Missing: {FACT_TRADE}"

In [ ]:
# helper functions
def _qident(col: str) -> str:
    """Quote a DuckDB identifier safely (minimal)."""
    return '"' + col.replace('"','""') + '"'

def duck_scan(path: Path) -> str:
    """
    DuckDB scan expression for CSV or Parquet.
    - .csv / .csv.gz -> read_csv_auto
    - .parquet       -> read_parquet
    """
    p = path.as_posix()
    suffix = path.suffix.lower()
    if suffix in [".csv", ".gz"]:   # adjust if you use .csv.gz explicitly
        return f"read_csv_auto('{p}')"
    elif suffix == ".parquet":
        return f"read_parquet('{p}')"
    else:
        raise ValueError(f"Unsupported file type for {path}")


def table_overview(path: Path):
    """Row count and head using DuckDB (fallback: pandas)."""
    if HAVE_DUCKDB:
        scan = duck_scan(path)
        rowcount = duckdb.sql(f"SELECT COUNT(*) FROM {scan}").fetchone()[0]
        head = duckdb.sql(f"SELECT * FROM {scan} LIMIT 5").df()
        schema = duckdb.sql(f"DESCRIBE SELECT * FROM {scan}").df()
        return rowcount, schema, head
    else:
        df = pd.read_csv(path, nrows=5000)
        rowcount = len(pd.read_csv(path, usecols=[0], engine="python"))
        schema = pd.DataFrame({"column": df.columns, "dtype(pandas)": [str(t) for t in df.dtypes]})
        head = df.head(5)
        return rowcount, schema, head

def distinct_report(path: Path, sample_n: int = 25):
    """
    Compute distinct count per column, plus up to `sample_n` example distinct values.
    Uses DuckDB when available (recommended).
    """
    if HAVE_DUCKDB:
        scan = duck_scan(path)
        cols = duckdb.sql(f"SELECT * FROM {scan} LIMIT 0").df().columns.tolist()
        recs = []
        for c in cols:
            qc = _qident(c)
            n_unique = duckdb.sql(f"SELECT COUNT(DISTINCT {qc}) FROM {scan}").fetchone()[0]
            # sample unique values (ordered; text sorts lexicographically)
            sample = duckdb.sql(f"SELECT DISTINCT {qc} AS v FROM {scan} ORDER BY {qc} NULLS FIRST LIMIT {sample_n}").df()["v"].tolist()
            recs.append({"column": c, "n_unique": n_unique, "sample_values": sample})
        return pd.DataFrame(recs).sort_values("column").reset_index(drop=True)
    else:
        # Fallback: pandas (loads whole file, OK for quick looks but slower)
        df = pd.read_csv(path)
        out = []
        for c in df.columns:
            vals = df[c].drop_duplicates()
            out.append({
                "column": c,
                "n_unique": int(vals.shape[0]),
                "sample_values": vals.sort_values(kind="stable").head(sample_n).tolist(),
            })
        return pd.DataFrame(out).sort_values("column").reset_index(drop=True)


In [ ]:
# fct_ip_flows_yearly overview
rows, schema, head = table_overview(FACT_ECON)
print(f"Rows (yearly): {rows}")
display(schema)
display(head)


In [ ]:
# distinct values per column (yearly)
uniq_yearly = distinct_report(FACT_ECON, sample_n=10000)
display(uniq_yearly)

In [ ]:
# overview of fct_TRADE
rows_c, schema_c, head_c = table_overview(FACT_TRADE)
print(f"Rows (by_class): {rows_c}")
display(schema_c)
display(head_c)